# Bonus — Model H without positional embeddings

Self-attention is permutation invariant: it scores every position against
every other with no term for *where* they are. Positional embeddings are what
break that symmetry. This notebook removes them and measures what is lost.

**The design is a matched comparison, not two numbers.** The ablation runs
the full 14,329 steps, matching the baseline exactly, and every claim is made at equal step
count, equal tokens seen, equal learning rate, equal seed. The only difference
between the two runs is `pos_encoding: learned` versus `none`.

| | Baseline | Ablation |
|:--|:--|:--|
| Config | `hindi/configs/model.yaml` | `hindi/configs/model_nopos.yaml` |
| Parameters | 25,379,840 | 25,117,696 |
| Positional embedding | learned, 512 × 512 | none |
| Everything else | — | identical |

**The learning rate is the trap.** At full length the schedules coincide by construction, but
`schedule.total_steps` is kept separate from `train.max_steps` so a shortened
run could not silently anneal the learning rate faster than the baseline, and
`src.eval.compare_ablation` **verifies** the two curves actually did coincide
rather than trusting the config.

**No new compute is needed for the baseline side** — `hindi/logs/train.jsonl`
already records the with-positions loss at every one of these steps.

**Runtime: GPU.** About 3.5 hours for training, plus ~25 minutes for the full evaluation suite.




## 1. Setup


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))


In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/24KaratAu'
WORK = '/content/phase2_ablation'

if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(f'{DRIVE_BASE}/code', WORK)
%cd {WORK}


In [ ]:
!apt-get install -y -qq fonts-indic 2>&1 | tail -2
!pip install -q pyyaml sentencepiece


### Confirm the ablation config differs in exactly one thing

An ablation is only interpretable if a single variable moved. This asserts it
before spending any GPU time.


In [ ]:
import yaml

base = yaml.safe_load(open('hindi/configs/model.yaml'))['architecture']
abl  = yaml.safe_load(open('hindi/configs/model_nopos.yaml'))['architecture']
diff = {k: (base.get(k), abl.get(k)) for k in set(base) | set(abl)
        if base.get(k) != abl.get(k)}
print('architecture differences (baseline -> ablation):', diff)
assert diff == {'pos_encoding': ('learned', 'none')}, diff

bt = yaml.safe_load(open('hindi/configs/train.yaml'))
at = yaml.safe_load(open('hindi/configs/train_nopos.yaml'))
assert at['optim'] == bt['optim'], 'optimiser settings must be identical'
assert at['seed'] == bt['seed'], 'seed must be identical'
assert at['train']['batch_size'] * at['train']['grad_accum'] == \
       bt['train']['batch_size'] * bt['train']['grad_accum'], 'tokens/step must match'
assert at['schedule']['total_steps'] == bt['train']['max_steps'], \
       'schedule horizon must equal the baseline run length'
print('optimiser, seed, tokens/step and schedule horizon all match the baseline')


## 2. Train (~3.5 hours)

Full 14,329 steps -- one epoch over the same 468,638,315 training tokens as the
graded model, so the ablation is comparable to Model H at its final state.

Checkpoints go to `hindi/checkpoints_nopos/` and the log to `train_nopos.jsonl`,
so this cannot touch the graded run's artifacts.

**If Colab disconnects, just re-run this cell.** The trainer finds
`checkpoint_latest.pt` in that directory and resumes from it -- model, optimizer,
scheduler, GradScaler and RNG. Checkpoints are written every 500 steps, so a drop
costs at most ~7 minutes of compute.

Check the first lines: parameters must read **25,117,696**, not 25,379,840. If it
reads the latter the position table was not removed -- stop and fix the config
rather than letting it run for three hours.


In [ ]:
!python -u -m src.train.trainer --config hindi/configs/train_nopos.yaml


## 3. Full Phase 2 evaluation suite on the ablated model

The brief requires the *full* suite on the ablated model — perplexity / BPB,
BLEU / chrF++ / ROUGE-L, diversity diagnostics, and attention — not just a
loss curve. These are the same modules used for the graded models, pointed at
the ablation checkpoint and `model_nopos.yaml`.

### 3a. Perplexity and bits-per-byte (~15 min)


In [ ]:
!python -u -m src.eval.intrinsic \
    --checkpoint {DRIVE_BASE}/hindi/checkpoints_nopos/checkpoint_best.pt \
    --model-config hindi/configs/model_nopos.yaml \
    --tokens-dir {DRIVE_BASE}/hindi/tokens \
    --language hindi --splits val,test \
    --out hindi/eval/intrinsic_nopos.json


### 3b. Generation and generation metrics

Same 32 held-out prompts, same seed, same decoding settings as the graded
model — so the two sets are directly comparable rather than merely similar.


In [ ]:
!python -u -m src.eval.generate \
    --checkpoint {DRIVE_BASE}/hindi/checkpoints_nopos/checkpoint_best.pt \
    --model-config hindi/configs/model_nopos.yaml \
    --tokenizer tokenizers/hindi/hindi_bpe_8k.model \
    --test-jsonl {DRIVE_BASE}/hindi/splits/test.jsonl \
    --language hindi \
    --n-prompts 32 --prompt-tokens 32 --max-new-tokens 128 --seed 1234 \
    --out hindi/samples/generations_nopos.json \
    --markdown hindi/samples/generations_nopos.md


In [ ]:
# Score both side by side: the graded model and the ablation.
!python -u -m src.eval.metrics \
    --generations hindi=hindi/samples/generations.json \
                  hindi_nopos=hindi/samples/generations_nopos.json \
    --out report/figures/generation_metrics_ablation.json \
    --markdown report/figures/generation_metrics_ablation.md


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('report/figures/generation_metrics_ablation.md').read()))


### 3c. What the ablated model actually writes

The metrics say whether it repeats itself; only reading it says whether it is
language.


In [ ]:
import json
g = json.load(open('hindi/samples/generations_nopos.json'))
b = json.load(open('hindi/samples/generations.json'))
for i in (0, 1):
    print('=' * 78)
    print('PROMPT   :', g['records'][i]['prompt_text'][:150])
    print('REFERENCE:', g['records'][i]['reference_text'][:200])
    print('BASELINE :', b['records'][i]['generations']['temp_1']['text'][:200])
    print('ABLATION :', g['records'][i]['generations']['temp_1']['text'][:200])


## 4. Attention structure without positions

The sharper prediction is here rather than in the loss. A *previous-token* head
-- the cleanest structure in the baseline, 96% of its mass on position *i-1* at
layer 4 -- should be unable to form at all, because "previous" is exactly what a
model with no position signal cannot identify.

Because the ablation runs the full 14,329 steps, this compares against the
committed baseline statistics in `hindi/eval/attention_stats.json`, which were
measured on `checkpoint_best.pt` at the same step with the same settings. Both
sides are therefore matched on training length as well as on everything else.


In [ ]:
!python -u -m src.eval.attention_analysis \
    --checkpoint {DRIVE_BASE}/hindi/checkpoints_nopos/checkpoint_best.pt \
    --model-config hindi/configs/model_nopos.yaml \
    --tokenizer tokenizers/hindi/hindi_bpe_8k.model \
    --tokens-dir {DRIVE_BASE}/hindi/tokens \
    --language hindi --batch-size 4 --n-batches 8 \
    --display-tokens 40 --heads 0,1,4,7 \
    --figdir hindi/figures_nopos \
    --out hindi/eval/attention_stats_nopos.json


## 5. Matched-step comparison

Aligns the runs to their shared steps, **verifies the learning-rate schedules
coincided**, and reports the loss gap and the attention contrast.



In [ ]:
import shutil, os
os.makedirs('hindi/logs', exist_ok=True)
shutil.copy(f'{DRIVE_BASE}/hindi/logs/train_nopos.jsonl',
            'hindi/logs/train_nopos.jsonl')

!python -u -m src.eval.compare_ablation \
    --baseline-log hindi/logs/train.jsonl \
    --ablation-log hindi/logs/train_nopos.jsonl \
    --baseline-attn hindi/eval/attention_stats.json \
    --ablation-attn hindi/eval/attention_stats_nopos.json \
    --language hindi --outdir report/figures \
    --out report/figures/ablation_nopos_hindi.json \
    --markdown report/figures/ablation_nopos_hindi.md




In [ ]:
from IPython.display import Image, display
for f in ('hindi_ablation_nopos_loss.png', 'hindi_ablation_nopos_prevtoken.png'):
    display(Image(f'report/figures/{f}'))


### The layer-4 heatmap, with and without positions

Left to right these are the same four heads of the same layer on the same
held-out passage. In the baseline, head 4 is an unbroken sub-diagonal stripe.


In [ ]:
from IPython.display import Image, display
print('BASELINE -- learned positions, step 14,329')
display(Image('hindi/figures/attention_layer4.png'))
print('ABLATION -- no positional embedding, step 14,329')
display(Image('hindi/figures_nopos/attention_layer4.png'))


## 6. Push results back to Drive



In [ ]:
import shutil, os
DEST = f'{DRIVE_BASE}/eval_outputs'
for src in ('report/figures', 'hindi/figures_nopos',
            'hindi/eval',
            'hindi/samples', 'hindi/logs'):
    if not os.path.exists(src):
        continue
    dst = f'{DEST}/{src}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'{src:22} -> {dst}  ({len(os.listdir(src))} files)')



